# Data Reading  Autoloader Incremental Loading

In [0]:
dbutils.widgets.text("file_name","")

In [0]:
p_file_name = dbutils.widgets.get("file_name")

In [0]:
df = (
    spark.readStream.format("cloudFiles")
    # autoloaders
    .option("cloudFiles.format", "csv")
    # for schema evolution
    .option(
        "cloudFiles.schemaLocation",
        f"abfss://bronze@azuredatabricksete.dfs.core.windows.net/checkpoint_{p_file_name}",
    ).load(f"abfss://source@azuredatabricksete.dfs.core.windows.net/{p_file_name}")
)

# Data Writing 

In [0]:
df.writeStream.format("parquet").outputMode("append").option(
    "checkpointLocation",
    f"abfss://bronze@azuredatabricksete.dfs.core.windows.net/checkpoint_{p_file_name}",  # should match the schema location --> both checkpoint and schema stored at the same location
).option(
    "path", f"abfss://bronze@azuredatabricksete.dfs.core.windows.net/{p_file_name}"
).trigger(
    once=True
).start()

In [0]:
df= spark.read.format("parquet").load(f"abfss://bronze@azuredatabricksete.dfs.core.windows.net/{p_file_name}")
display(df)